# 1. Header, Citations, Purpose & Topic Bridges

### Theoretical Significance & Research Lineage
Visual autoregression historically relied on 1D raster-scan traversal (ImageGPT, DALL-E), which enforced artificial unidirectional causality on naturally 2D isotropic visual fields. Chang et al. (2022) introduced **MaskGIT** (Masked Generative Image Transformer), replacing sequential scanning with iterative bidirectional unmasking using cosine schedules. Tian et al. (2024) introduced **VAR** (Visual AutoRegressive Modeling), decomposing visual generation into coarse-to-fine next-scale autoregression. Li et al. (2024) introduced **MAR** (Masked Autoregressive Models), bypassing discrete vector quantization errors by optimizing continuous diffusion loss heads on masked continuous VAE representations.

### Purpose & Target Architecture
This notebook builds from first principles:
1. MaskGIT cosine masking schedule and iterative confidence-based unmasking.
2. VAR next-scale spatial expansion with scale-residual projections.
3. MAR masked continuous diffusion loss over dynamically masked patch tokens.

# 2. Mathematical Foundations & Governing Equations

### MaskGIT Cosine Schedule
Training masks visual tokens with schedule $\gamma(r)$:
$$\gamma(r) = \cos\left(\frac{\pi}{2} r\right), \quad r \in [0, 1]$$
The model optimizes categorical cross-entropy on masked positions:
$$\mathcal{L}_{\text{MaskGIT}}(\theta) = -\sum_{i=1}^N M_i \log p_\theta(y_i | Y \odot (1 - M) + \text{[MASK]} \odot M)$$

### VAR Next-Scale Factorization
Rather than raster order, VAR decomposes generation into resolution scales $(R_1, \dots, R_K)$:
$$p(R_1, \dots, R_K) = \prod_{k=1}^K p(R_k | R_1, \dots, R_{k-1})$$
Within scale $R_k$, tokens are generated concurrently with bidirectional self-attention.

### MAR Masked Continuous Diffusion Loss
Operating on continuous latents $Z \in \mathbb{R}^{N \times D}$, masked tokens $M_i = 1$ satisfy:
$$\mathcal{L}_{\text{MAR}}(\theta) = \frac{\sum_{i=1}^N M_i \| v_\theta(z_i, t) - v_i^* \|^2}{\sum_{i=1}^N M_i \cdot D + \epsilon}$$

In [ ]:
# 3. Deterministic Setup & Micro-Data Generator
import math
import torch
import torch.nn.functional as F

torch.manual_seed(42)

# Synthetic visual token sequence: Batch=4, 4 patches (2x2 grid), Dim=16
B, L, D = 4, 4, 16
x_tokens = torch.randn(B, L, D)
mask_token = torch.randn(1, 1, D) * 0.5

# Step ratio r
r = torch.linspace(0.0, 1.0, 5)
print(f"Setup complete: tokens shape={x_tokens.shape}, mask_token shape={mask_token.shape}")
print(f"Timestep ratios: {r.tolist()}")

In [ ]:
# 4. First-Principles Implementation & Native Parity

def maskgit_cosine_schedule(r: torch.Tensor) -> torch.Tensor:
    return torch.cos(0.5 * math.pi * r)

def maskgit_apply_mask(x: torch.Tensor, mask: torch.Tensor, mask_token: torch.Tensor) -> torch.Tensor:
    return (1.0 - mask) * x + mask * mask_token

def var_coarse_to_fine_upsample(coarse_map: torch.Tensor, repeats: int, w_scale: torch.Tensor, b_scale: torch.Tensor) -> torch.Tensor:
    upsampled = coarse_map.repeat_interleave(repeats, dim=1)
    residual = F.gelu(upsampled @ w_scale + b_scale)
    return upsampled + residual

def mar_masked_diffusion_loss(v_pred: torch.Tensor, v_target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    sq_err = (v_pred - v_target) ** 2
    masked_sq_err = sq_err * mask
    loss = torch.sum(masked_sq_err) / (torch.sum(mask) * v_pred.shape[-1] + 1e-6)
    return loss.reshape(1)

# Forward pass
gamma = maskgit_cosine_schedule(r)
mask = (torch.rand(B, L, 1) > 0.5).float()
masked_x = maskgit_apply_mask(x_tokens, mask, mask_token)

# VAR expansion: coarse 1x1 to fine 2x2 (repeats=4)
coarse_1x1 = torch.randn(B, 1, D)
w_scale = torch.randn(D, D) * 0.2
b_scale = torch.randn(D) * 0.1
fine_2x2 = var_coarse_to_fine_upsample(coarse_1x1, 4, w_scale, b_scale)

# MAR loss
v_target = torch.randn(B, L, D)
v_pred = masked_x.clone()
mar_loss = mar_masked_diffusion_loss(v_pred, v_target, mask)

print(f"MaskGIT gamma values: {gamma.tolist()}")
print(f"VAR fine scale shape: {fine_2x2.shape}")
print(f"MAR masked diffusion loss: {mar_loss.item():.4f}")

In [ ]:
# 5. Quantitative Invariants & Parity Verification

# Invariant 1: Cosine schedule boundary values: gamma(0) = 1, gamma(1) = 0
assert abs(maskgit_cosine_schedule(torch.tensor(0.0)).item() - 1.0) < 1e-6, "gamma(0) must equal 1.0"
assert abs(maskgit_cosine_schedule(torch.tensor(1.0)).item() - 0.0) < 1e-6, "gamma(1) must equal 0.0"

# Invariant 2: When mask == 0, masked_x == x_tokens; when mask == 1, masked_x == mask_token
zeros_mask = torch.zeros(B, L, 1)
ones_mask = torch.ones(B, L, 1)
assert torch.allclose(maskgit_apply_mask(x_tokens, zeros_mask, mask_token), x_tokens, atol=1e-6), "Zero mask must preserve tokens"
assert torch.allclose(maskgit_apply_mask(x_tokens, ones_mask, mask_token), mask_token.expand_as(x_tokens), atol=1e-6), "One mask must replace with mask token"

# Invariant 3: MAR loss on unmasked tokens must be strictly zero
v_perfect_masked = v_target.clone()
# Perturb unmasked tokens only
v_perturbed = v_perfect_masked + (1.0 - mask) * 10.0
zero_loss = mar_masked_diffusion_loss(v_perturbed, v_target, mask)
assert zero_loss.item() < 1e-5, f"Unmasked token perturbations must not contribute to MAR loss: got {zero_loss.item():.2e}"
print("All visual autoregressive invariants verified successfully!")

In [ ]:
# 6. Performance Profile & Diagnostic Visualizations
import time

# Benchmark iterative unmasking across 12 steps vs 256 raster steps
start = time.perf_counter()
x_curr = mask_token.repeat(B, L, 1)
for step in range(12):
    step_r = torch.tensor(step / 12.0)
    step_gamma = maskgit_cosine_schedule(step_r)
    # Simulate unmasking step
    active_mask = (torch.rand(B, L, 1) < step_gamma).float()
    x_curr = maskgit_apply_mask(x_tokens, active_mask, mask_token)
maskgit_time = time.perf_counter() - start

# Simulate 256 sequential raster tokens
start = time.perf_counter()
dummy_raster = torch.zeros(B, 1, D)
for _ in range(16):  # 16-step coarse raster
    dummy_raster = dummy_raster + 0.01
raster_time = time.perf_counter() - start

print(f"MaskGIT 12-step parallel unmasking latency: {maskgit_time*1000:.2f} ms")
print(f"Sequential raster token latency: {raster_time*1000:.2f} ms")

In [ ]:
# 7. Pathological Misuse vs. Corrected Implementation

# Pathological Misuse: Linear schedule de-masks too quickly early in generation
def linear_schedule_bad(r: torch.Tensor) -> torch.Tensor:
    return 1.0 - r

r_eval = torch.linspace(0.0, 1.0, 10)
bad_gamma = linear_schedule_bad(r_eval)
good_gamma = maskgit_cosine_schedule(r_eval)

# Compare mid-point mask retention at r = 0.5
mid_idx = 5
print(f"At r=0.5: Linear schedule retains {bad_gamma[mid_idx].item()*100:.1f}% masks (too little global context)")
print(f"At r=0.5: Cosine schedule retains {good_gamma[mid_idx].item()*100:.1f}% masks (preserves stability)")

# Pathological Misuse 2: Unnormalized MAR loss causing gradient explosion
def unnormalized_mar_bad(v_pred: torch.Tensor, v_target: torch.Tensor, mask: torch.Tensor):
    # Divides by batch*seq instead of active masks
    return torch.mean((v_pred - v_target) ** 2 * mask)

# Corrected: Strictly normalized by active mask tokens
bad_loss = unnormalized_mar_bad(v_pred, v_target, mask)
good_loss = mar_masked_diffusion_loss(v_pred, v_target, mask)
print(f"Unnormalized loss (diluted by sparse masks): {bad_loss.item():.4f}")
print(f"Properly normalized mask loss: {good_loss.item():.4f}")

# 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

### Core Insights
1. **Next-Scale Paradigm**: VAR mirrors natural hierarchical visual perception by generating coarse layout first and fine details second, eliminating 1D raster scan limitations.
2. **Iterative Cosine Schedule**: The concave cosine schedule allows MaskGIT to resolve global composition during early cautious iterations before finalizing localized textures.
3. **Continuous Visual Modeling**: MAR unites autoregression and diffusion on continuous latent manifolds, overcoming discrete codebook collapse.